# Harness Bench

**A very small benchmark to find the harness you need.**

Choose the project's `.venv` Python kernel. This notebook opens saved results by default. It never launches a campaign until you set `run_new_campaign = True`. Each work score retains errors, timeouts and unexecuted trials in its denominator. The startup check is reported separately.


In [ ]:
from pathlib import Path
import importlib
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/local_comparison.py").is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "scripts"))
import bench
import local_comparison
import scripts.campaign_results as campaign_results
importlib.reload(campaign_results)
importlib.reload(bench)
importlib.reload(local_comparison)
print("Python kernel", sys.executable)

## Choose the experiment

`demo` contains two tasks, `smoke` three, `quick` six and `full` nine. Quick covers Python, data science, research and general work, plus a separate startup check. The preview below shows the exact task instructions and campaign size before anything runs.

Leave `run_new_campaign = False` to inspect saved results. Set `campaign_id` to a saved timestamp, or leave it as `None` for the newest local campaign. A new campaign records requested limits and available Ollama runtime metadata. Actual context observations can be absent when the server unloads a model.


In [ ]:
model = "lfm3:1.7b"
harnesses = ["pi", "opencode", "droid", "dsh"]
suite = "quick"
attempts = 1
timeout_sec = 180
context_window = 32768
max_output_tokens = 8192
diagnostic_task = None  # Choose one task name, or auto-select a work task.

run_new_campaign = False  # Change to True when you want to launch.
campaign_id = None

In [ ]:
from IPython.display import HTML, display
import html
import tomllib

plan = bench.build_plan(suite, harnesses, dict.fromkeys(harnesses, model), attempts, local=True)
settings, _ = bench.read_config()
task_names = settings["suites"][suite]
work_count = sum(name != "startup" for name in task_names)
print(f"Preview only: {plan['runs']} sequential trials, {work_count} work tasks per harness.")
print(f"Agent time limit total: {plan['runs'] * timeout_sec / 60:.0f} minutes if every trial times out.")
print("Setup and grading add time. Local inference has no provider token charge; hardware uses energy.")
for name in task_names:
    folder = root / "tasks" / name
    metadata = tomllib.loads((folder / "task.toml").read_text())["metadata"]
    instructions = (folder / "instruction.md").read_text()
    display(HTML("<details><summary><strong>" + html.escape(name) + "</strong> · "
                 + html.escape(metadata["category"]) + "</summary><pre style='white-space:pre-wrap'>"
                 + html.escape(instructions) + "</pre></details>"))


## Prepare the notebook

This cell installs matplotlib in the selected Python environment if needed. Docker and Ollama preparation happens only when you request a new campaign. Start their applications first. Before a new campaign, it also updates stale execution libraries from the lockfile. Existing reports do not require an execution-library update.

In [ ]:
import importlib.util
import json
import shutil
import subprocess
import urllib.request

installer = (["uv", "pip", "install", "--python", sys.executable]
             if shutil.which("uv") else [sys.executable, "-m", "pip", "install"])
def install_notebook_packages(arguments):
    if installer[0] == sys.executable and importlib.util.find_spec("pip") is None:
        subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)
    subprocess.run(installer + arguments, check=True)

if importlib.util.find_spec("matplotlib") is None:
    install_notebook_packages(["matplotlib>=3.9,<4"])

if run_new_campaign:
    try:
        bench.runtime_versions()
    except ValueError:
        print("Updating execution dependencies before Docker and model setup.")
        install_notebook_packages(["-r", str(root / "requirements.lock")])
        bench.runtime_versions()
    subprocess.run(["docker", "info"], check=True, capture_output=True)
    image = subprocess.run(
        ["docker", "image", "inspect", "harness-bench-base:1"], capture_output=True
    )
    if image.returncode:
        subprocess.run(["docker", "build", "-t", "harness-bench-base:1", "docker"],
                       cwd=root, check=True)
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as response:
        installed_models = json.load(response)["models"]
    if model not in {item["name"] for item in installed_models}:
        subprocess.run(["ollama", "pull", model], check=True)

## Run or select a campaign

The progress bar counts finished trials. While a trial runs, it shows the latest available event and the time since a native log changed. Empty logs are reported as awaiting events. Interrupt the kernel to cancel. Once the campaign has been created, recoverable failures and cancellations still return its partial report path.

In [ ]:
if run_new_campaign:
    campaign = local_comparison.run_comparison(
        model=model, harnesses=harnesses, suite=suite, attempts=attempts,
        timeout_sec=timeout_sec, context_window=context_window,
        max_output_tokens=max_output_tokens,
    )
else:
    available = local_comparison.saved_campaigns()
    campaign = (root / "runs/campaigns" / campaign_id if campaign_id
                else next(iter(available), None))
if campaign is None:
    raise ValueError("No local campaign saved yet. Set run_new_campaign = True to start.")
print("Campaign", campaign)
rows = campaign_results.load_rows(campaign)

## Read the evidence

The table and overview score **work tasks only**. Startup is a health check shown separately. A success requires every required check to pass. Timeouts, technical errors and unfinished scheduled work count against that work score. Unknown measurements stay unknown, with observed counts shown alongside cost and runtime.

The thin lines are exploratory 95% Wilson intervals for this set of trials. Small samples produce wide intervals even at 0% or 100%. Task attempts can be dependent, so these intervals do not establish a general harness ranking. Compare the same model, task mix and limits, then inspect the outputs.

Category and task heatmaps show each denominator explicitly. Gray N/A means no applicable trial or unavailable diagnostic. The diagnostics panel covers one task selected with `diagnostic_task`; the panel title identifies it. Diagnostic percentages use only recorded checks and are not partial-credit grades. Older retrieval campaigns can recover limited public diagnostics from saved answers; their hidden checks remain unavailable.


In [ ]:
from IPython.display import HTML, display
import matplotlib.pyplot as plt

print("Campaign:", campaign.name)
display(HTML(campaign_results.summary_html(rows)))
figure = campaign_results.plot_results(rows, diagnostic_task=diagnostic_task)
display(figure)
plt.close(figure)
task_figure = campaign_results.plot_task_results(rows)
display(task_figure)
plt.close(task_figure)

# Optional exports. Uncomment to save locally; do not publish private trial outputs.
# figure.savefig(campaign / "overview.png", dpi=160, bbox_inches="tight")
# task_figure.savefig(campaign / "tasks.png", dpi=160, bbox_inches="tight")


## Matched comparisons

These counts compare the same model tag, task and attempt. A win means one harness passed and the other did not. A tie can mean both passed or both failed. Unexecuted and incomplete trials are excluded from pairs and counted as unmatched rows. Different model tags are never paired. Identical tags are meaningful only when weights, runtime settings and task versions also match; the campaign manifest records those details.

This is a description of the observed trials, with no overall winner declared. Check the per-task results and failure reasons before choosing a harness.


In [ ]:
pairs = campaign_results.paired_outcomes(rows)
if pairs:
    columns = ["left", "right", "model", "matched", "left_wins", "right_wins", "ties", "unmatched"]
    header = "".join("<th>" + html.escape(key.replace("_", " ")) + "</th>" for key in columns)
    body = "".join("<tr>" + "".join("<td>" + html.escape(str(pair[key])) + "</td>"
                                  for key in columns) + "</tr>" for pair in pairs)
    display(HTML('<table class="hb-table"><thead><tr>' + header + "</tr></thead><tbody>"
                 + body + "</tbody></table>"))
else:
    print("No configurations with the same model tag to pair.")


## Inspect each trial

Use the logs folder to inspect native events, exception.txt, verifier output and generated programs. A timeout may still have partial files. Review the error and diagnostics before comparing harnesses.

In [ ]:
import html
import os
from urllib.parse import quote
from IPython.display import HTML

columns = ["harness", "task", "attempt", "status", "reward", "agent_seconds"]
header = "".join("<th>" + html.escape(key) + "</th>" for key in columns)
body = []
for row in rows:
    values = "".join("<td>" + html.escape(str("unknown" if row.get(key) in (None, "") else row[key])) + "</td>"
                     for key in columns)
    result = Path(row["result"]) if row.get("result") else None
    if result and result.exists():
        url = os.path.relpath(result.parent, Path.cwd())
        link = '<a href="' + html.escape(quote(url), quote=True) + '">Open trial files</a>'
    else:
        link = "No trial files"
    body.append("<tr>" + values + "<td>" + link + "</td></tr>")
display(HTML("<table><thead><tr>" + header + "<th>Logs</th></tr></thead><tbody>"
             + "".join(body) + "</tbody></table>"))
for row in rows:
    checks = campaign_results.diagnostic_checks(row)
    print(row["harness"], row["task"], checks)
    if row.get("error_message"):
        print(row["error_message"])